In [ ]:
import pandas as pd
import json
import numpy as np
from datetime import datetime


import sqlalchemy
from sqlalchemy import create_engine, text


import psycopg2
from psycopg2.extras import execute_values

In [ ]:
# === CARGAR DATOS HISTÓRICOS EN NOTEBOOK ===

with open('C:/Users/crisr/dev/rawg-aws-ml-analytics/notas_cris/00_data/extraccion_historica.json', 'r') as f:
    datos = json.load(f)

print(f"Total de juegos cargados: {len(datos)}")

In [ ]:
# === TRANSFORMAR DATOS BRUTOS A DATAFRAMES ===

# Transformación de todos los datos:
print("Iniciando transformación de los datos en DataFrames limpios...")
resultado = transformar_datos_completo(datos, verbose = False)
print("Transformación completada exitosamente")

resultado.keys()

In [ ]:
# === LIMPIAR DataFrame GAMES (corrige el dtype del DataFrame) -> PREPARAR PARA LA CARGA DE DATOS ===
df = resultado["games"].copy()

# esrb_id: float -> Int64 nullable
if "esrb_id" in df.columns:
    df["esrb_id"] = pd.to_numeric(df["esrb_id"], errors="coerce").astype("Int64")

# columnas year-month como string nullable
for c in ["released_ym", "updated_ym"]:
    if c in df.columns:
        df[c] = df[c].astype("string")

resultado["games"] = df


In [ ]:
# === DEFINIR LA FUNCIÓN ADAPTACIÓN DE DATOS PARA LA INSERCIÓN ===

def df_to_python_records(df: pd.DataFrame):
    # Convierte pd.NA, NAN, None en None. Evita el error "can´t adapt 'NAType'" al insertar los datos
    # Adapta los datos para psycopg2
    
    df2 = df.copy()

    def conv(x):
        # pd.NA / NaN -> None
        if x is None or pd.isna(x):
            return None

        # numpy scalars -> python scalars
        if isinstance(x, (np.integer,)):
            return int(x)
        if isinstance(x, (np.floating,)):
            return float(x)
        if isinstance(x, (np.bool_,)):
            return bool(x)

        return x

    # Celda a celda (evita NAType)
    return [tuple(conv(v) for v in row) for row in df2.to_numpy(dtype=object)]


In [ ]:
# Esquema creado en pgADMIN4
schema_sql = """
CREATE SCHEMA IF NOT EXISTS rawg;

-- =========================
-- DIMENSIONES / CATÁLOGOS
-- =========================

CREATE TABLE IF NOT EXISTS rawg.esrb_ratings (
  esrb_id   INT PRIMARY KEY,
  esrb_name VARCHAR(100) NOT NULL
);

CREATE TABLE IF NOT EXISTS rawg.platforms (
  platform_id   INT PRIMARY KEY,
  platform_name VARCHAR(200) NOT NULL
);

CREATE TABLE IF NOT EXISTS rawg.genres (
  genre_id          INT PRIMARY KEY,
  genre_name        VARCHAR(100) NOT NULL,
  genre_games_count BIGINT
);

CREATE TABLE IF NOT EXISTS rawg.stores (
  store_id   INT PRIMARY KEY,
  store_name VARCHAR(200) NOT NULL
);

CREATE TABLE IF NOT EXISTS rawg.tags (
  tag_id          BIGINT PRIMARY KEY,
  tag_name        VARCHAR(200) NOT NULL,
  tag_language    VARCHAR(50),
  tag_games_count BIGINT
);

-- =========================
-- HECHO PRINCIPAL
-- =========================

CREATE TABLE IF NOT EXISTS rawg.games (
  game_id           BIGINT PRIMARY KEY,
  game_name         VARCHAR(500) NOT NULL,
  tba               BOOLEAN,
  released_ym       CHAR(7),
  updated_ym        CHAR(7),

  game_rating        NUMERIC(4,2),
  ratings_count      BIGINT,
  game_added         BIGINT,
  playtime           INTEGER,
  suggestions_count  BIGINT,

  esrb_id           INT NULL,
  created_at        TIMESTAMP DEFAULT CURRENT_TIMESTAMP,

  CONSTRAINT fk_games_esrb
    FOREIGN KEY (esrb_id) REFERENCES rawg.esrb_ratings(esrb_id)
);

CREATE TABLE IF NOT EXISTS rawg.games_status (
  game_id BIGINT PRIMARY KEY
    REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  yet     BIGINT NOT NULL DEFAULT 0,
  owned   BIGINT NOT NULL DEFAULT 0,
  beaten  BIGINT NOT NULL DEFAULT 0,
  toplay  BIGINT NOT NULL DEFAULT 0,
  dropped BIGINT NOT NULL DEFAULT 0,
  playing BIGINT NOT NULL DEFAULT 0
);

-- Distribución de ratings por juego (típicamente 4 filas por juego)
CREATE TABLE IF NOT EXISTS rawg.ratings_distribution (
  game_id        BIGINT NOT NULL,
  ratingd_id     INT NOT NULL,          -- viene de la API (NO SERIAL)
  ratingd_title  VARCHAR(50),
  ratingd_count  BIGINT DEFAULT 0,
  ratingd_percent NUMERIC(6,2) DEFAULT 0.0,
  PRIMARY KEY (game_id, ratingd_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE
);

-- =========================
-- RELACIONES N:M (BRIDGES)
-- =========================

CREATE TABLE IF NOT EXISTS rawg.game_platforms (
  game_id     BIGINT NOT NULL,
  platform_id INT NOT NULL,
  released_at DATE NULL,   -- o CHAR(10) si te llega como string
  PRIMARY KEY (game_id, platform_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (platform_id) REFERENCES rawg.platforms(platform_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS rawg.game_genres (
  game_id  BIGINT NOT NULL,
  genre_id INT NOT NULL,
  PRIMARY KEY (game_id, genre_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (genre_id) REFERENCES rawg.genres(genre_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS rawg.game_stores (
  game_id  BIGINT NOT NULL,
  store_id INT NOT NULL,
  PRIMARY KEY (game_id, store_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (store_id) REFERENCES rawg.stores(store_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS rawg.game_tags (
  game_id BIGINT NOT NULL,
  tag_id  INT NOT NULL,
  PRIMARY KEY (game_id, tag_id),
  FOREIGN KEY (game_id) REFERENCES rawg.games(game_id) ON DELETE CASCADE,
  FOREIGN KEY (tag_id) REFERENCES rawg.tags(tag_id) ON DELETE CASCADE
);

-- =========================
-- ÍNDICES
-- =========================
CREATE INDEX IF NOT EXISTS ix_games_released ON rawg.games(released_ym);
CREATE INDEX IF NOT EXISTS ix_games_esrb_id ON rawg.games(esrb_id);
CREATE INDEX IF NOT EXISTS ix_ratings_distribution_game ON rawg.ratings_distribution(game_id);
CREATE INDEX IF NOT EXISTS ix_game_platforms_platform ON rawg.game_platforms(platform_id);
CREATE INDEX IF NOT EXISTS ix_game_genres_genre ON rawg.game_genres(genre_id);
CREATE INDEX IF NOT EXISTS ix_game_stores_store ON rawg.game_stores(store_id);
CREATE INDEX IF NOT EXISTS ix_game_tags_tag ON rawg.game_tags(tag_id);
"""

In [ ]:
# === DEFINIR LA FUNCIÓN CARGA ===


def upsert_to_table(
    df: pd.DataFrame,
    table: str,
    database: str,
    pk_cols: list[str],
    schema: str = "rawg",
    do_update: bool = True,
    host: str = "localhost",
    user: str = "postgres",
    password: str = "sql123",
    port: int = 5432,
    page_size: int = 5000
):
    """
    Inserta un DataFrame en PostgreSQL con ON CONFLICT.
    - do_update=True  -> UPSERT (update columnas no PK)
    - do_update=False -> DO NOTHING (ideal para tablas bridge)
    """
    if df is None or df.empty:
        print(f"[SKIP] {schema}.{table} (vacío)")
        return 0

    # 1) deduplicar por PK para que no rompa dentro del propio DF
    df = df.drop_duplicates(subset=pk_cols).copy()

    # 2) NaN -> NULL
    df = df.where(pd.notnull(df), None)

    cols = list(df.columns)
    rows = df_to_python_records(df)

    col_list = ", ".join(cols)
    pk_list = ", ".join(pk_cols)

    # columnas a actualizar
    update_cols = [c for c in cols if c not in pk_cols]

    if do_update and update_cols:
        set_clause = ", ".join([f"{c}=EXCLUDED.{c}" for c in update_cols])
        conflict = f"ON CONFLICT ({pk_list}) DO UPDATE SET {set_clause}"
    else:
        conflict = f"ON CONFLICT ({pk_list}) DO NOTHING"

    sql = f"INSERT INTO {schema}.{table} ({col_list}) VALUES %s {conflict};"

    db = psycopg2.connect(
        host=host, user=user, password=password, port=port, database=database
    )
    try:
        cursor = db.cursor()
        execute_values(cursor, sql, rows, page_size=page_size)
        db.commit()
        cursor.close()
        print(f"[OK] {schema}.{table} <- {len(df)} filas (UPSERT)")
        return len(df)
    finally:
        db.close()



In [ ]:
# === CONEXIÓN A POSTGRESQL LOCAL (pgAdmin4) ===

host = "localhost"
user = "postgres"
password = "sql123"
database = "rawg_db_local"
port = 5432

# Crear la conexión
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")

# Abrir la conexión
connection = engine.connect()

# Cerrar la conexión
connection.close()

In [ ]:
# === CARGAR DIMENSIONES Y GAMES ===

# Dimensiones tablas independientes
upsert_to_table(resultado['esrb_ratings'], 'esrb_ratings',"rawg_db_local", ["esrb_id"])
upsert_to_table(resultado['platforms'], 'platforms', "rawg_db_local",["platform_id"])
upsert_to_table(resultado['genres'], 'genres', "rawg_db_local",["genre_id"])
upsert_to_table(resultado['stores'], 'stores', "rawg_db_local",["store_id"])
upsert_to_table(resultado['tags'], 'tags',"rawg_db_local", ["tag_id"])
    
# Tabla principal    
upsert_to_table(resultado['games'], 'games', "rawg_db_local",["game_id"])


In [ ]:
# Leer game_id realmente existentes en BD
games_in_db = set(
    pd.read_sql(
        "SELECT game_id FROM rawg.games;",
        engine
    )["game_id"].tolist()
)

print(f"Games válidos en BD: {len(games_in_db)}")

# Filtrar DataFrames dependientes
dependientes = [
    "ratings_distribution",
    "game_platforms",
    "game_genres",
    "game_stores",
    "game_tags",
    "games_status",
]

for key in dependientes:
    df = resultado[key]
    antes = len(df)
    resultado[key] = df[df["game_id"].isin(games_in_db)].copy()
    despues = len(resultado[key])
    print(f"{key}: {antes} → {despues}")


In [ ]:
# Dimensiones Tablas dependientes
upsert_to_table(resultado['games_status'], 'games_status', "rawg_db_local", ["game_id"])
upsert_to_table(resultado['ratings_distribution', 'ratings_distribution', "rawg_db_local", ["game_id","ratingd_id"])
    
# Dimensiones Tablas bridge
upsert_to_table(resultado['game_platforms'], 'game_platforms', "rawg_db_local", ["game_id","platform_id"], do_update = False)
upsert_to_table(resultado['game_genres'], 'game_genres', "rawg_db_local", ["game_id","genre_id"], do_update = False)
upsert_to_table(resultado['game_stores'], 'game_stores', "rawg_db_local", ["game_id","store_id"], do_update = False)
upsert_to_table(resultado['game_tags'], 'game_tags', "rawg_db_local", ["game_id","tag_id"], do_update = False)
